# レッスン16 - Microsoft Foundryによるスケーラブルエージェントの展開

このノートブックでは、架空の企業<strong>Contoso</strong>のための<strong>本番対応のカスタマーサポートエージェント</strong>を構築します。前のレッスンと異なり、ここでのポイントはエージェントの推論ループではなく、スケールして安全に実行できるエージェントを支える周辺のすべてです：

1. <strong>ツール呼び出し</strong> — 注文の照会とチケット作成。
2. **RAG** — 知識ベースからのポリシー回答。
3. <strong>メモリ</strong> — 会話をまたいで顧客を記憶。
4. <strong>モデルルーティング</strong> — 単純なリクエストは小さなモデルへ、複雑なものは大きなモデルへ送信。
5. <strong>レスポンスキャッシュ</strong> — モデル呼び出しなしで繰り返し質問に対応。
6. <strong>人間の承認</strong> — 一定額以上の返金は署名確認のため保留。
7. <strong>評価ゲート</strong> — オフラインのテストセットで不適切なリリースをブロック。
8. <strong>可観測性</strong> — すべてのリクエストに対するOpenTelemetryトレーシング。

各セクションは独立して実行可能です。すべての行を読み進めてください — 本番用の原始的な構成は故意に小さく保たれています。


## セットアップ

このノートブックを実行する前に、以下を確認してください：

1. **展開されたチャットモデルを持つ Microsoft Foundry プロジェクト**（例：`gpt-5-mini`）。
2. **Azure CLI にログイン済み** — ターミナルで `az login` を実行してください。
3. **必要な環境変数を設定済み：**
   - `AZURE_AI_PROJECT_ENDPOINT` — あなたの Microsoft Foundry プロジェクトのエンドポイント。
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — 展開されたモデルの名前。

RAG セクションでは、`AZURE_SEARCH_SERVICE_ENDPOINT` と `AZURE_SEARCH_API_KEY` が設定されているときに **Azure AI Search** を使用し、設定されていない場合はインメモリ検索にフォールバックして、Search リソースなしでもノートブックが実行できるようにしています。


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. ツール

本番用ツールは実際のシステムに対して実際の作業を行います。ここでは単純なPython関数を使って、注文データベースとチケット発行システムをシミュレートします。`@tool`デコレータはそれらをエージェントに公開します。

`issue_refund`が一定額を超える返金に対して`approval_mode="always_require"`を使っていることに注目してください—これは後で導入する人間が介在するプリミティブです。


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — ポリシー ナレッジベース

ポリシーに関する質問（「返品期間はどのくらいですか？」など）は、モデルの記憶ではなく、権威ある情報源から回答する必要があります。小さなナレッジベースを検索ツールとしてラップしています。

本番環境では **Azure AI Search** を使用しますが、ここではノートブックがどこでも実行できるようにインメモリのキーワード検索を提供し、環境変数が存在すると自動的に Azure AI Search に切り替わるようにしています。


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. メモリ

誰と話しているかを忘れるサポート担当者は良いサポート担当者ではありません。私たちは顧客ごとに小さなプロファイルストアを維持し、その要約をエージェントの指示に注入します。本番環境ではこれはメモリサービスです（レッスン13を参照してください）；ここでは辞書でパターンを可視化しています。


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. モデルルーティングとレスポンスキャッシュ

1つのリクエストハンドラーに接続された2つのコストレバー：

- <strong>ルーティング</strong>: リクエストに小さいモデルか大きいモデルのどちらが必要かを決定する安価なヒューリスティック分類器。
- <strong>キャッシュ</strong>: 正規化された繰り返し質問はモデル呼び出しなしでキャッシュから直接提供される。

ここでの分類器はあえて単純にしています。本番環境ではトラフィックに対して検証し、FoundryのModel Routerに置き換えることができます。


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. エージェント、人間の承認、および可観測性

ここで、上記のツールからエージェントを組み立て、各リクエストを OpenTelemetry スパンでラップします。`handle_support_request` 関数は本番リクエストハンドラーです: キャッシュ → ルート → トレース → 実行 → キャッシュ。

人間の承認はフレームワークによって処理されます: `issue_refund` が `approval_mode="always_require"` であるため、実行は一時停止し、明示的に解決する承認リクエストが表示されます。


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. 評価ゲート

これはレッスンのリリースゲートです。オフラインのテストセットがエージェントを評価し、合格率が閾値を超えた場合にのみデプロイが進みます。ここでのスコアラーはノートブックを自己完結型に保つための単純なキーワード重複チェックであり、本番ではLLMを審査員として使うか、フレームワークの評価者を使用します（レッスン10を参照）。


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## 組み合わせる：シミュレートされたリリース

下のセルはこのレッスンで説明している全体のループを示しています：評価ゲートを実行し、合格した場合にのみ「デプロイ」します。これは、Foundry Agent Service にエージェントバージョンを昇格させる前にCIで実行するパターンです。


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## サマリー

あなたはすべての運用上の懸念が組み込まれた本番対応可能なカスタマーサポートエージェントを組み立てました:

- **ツール、RAG、およびメモリ** はエージェントに能力とコンテキストを与えます。
- <strong>モデルルーティングとキャッシュ</strong> は遅延とコストを管理します。
- <strong>人間の承認</strong> は大きな返金などのリスクの高い行動を守ります。
- <strong>評価ゲート</strong> は悪いリリースを出荷前に阻止します。
- <strong>トレーシング</strong> はすべてのリクエストを観測可能にします。

### チャレンジ

このエージェントを拡張してください:

1. <strong>複数モデルのサポート</strong> — 第三の「推論」層を追加し、エスカレーションや苦情をそこにルーティングします。
2. <strong>評価ゲートの追加</strong> — `TEST_CASES` を拡張して返金承認のシナリオを含め、ゲートが回帰を検出することを確認します。
3. <strong>コスト認識ルーティングの追加</strong> — リクエストごとに推定コスト（小・大・キャッシュ）を追跡し、混在クエリのバッチ後にコストレポートを出力します。

次のレッスンでは、逆の道をたどり、Microsoft Foundry Local と Qwen で完全に自分のマシン上でエージェントを実行します。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
